In [101]:
import numpy as np
from matplotlib import pyplot as plt

import casadi as cs
from liecasadi import SO3, SO3Tangent
from scipy.spatial.transform import Rotation

np.set_printoptions(precision=3, suppress=True)



In [ ]:
np.random.seed(42)

gyro_frame = Rotation.random().as_matrix()
a = acc0 = np.array([0,0,-1])

N = 20
dt = 1
noise_sigma = 0.2
gyros = []
true_gyros = []
accs = []
for i in range(N):
    g = np.random.randn(3)
    R = SO3Tangent(g * dt) + SO3.Identity()
    a = R.as_matrix() @ a
    a = a.toarray().T[0]
    accs.append(a + np.random.randn(3) * noise_sigma)
    gyros.append(gyro_frame @ g + np.random.randn(3) * noise_sigma)
    true_gyros.append(g)

accs = np.array(accs)
gyros = np.array(gyros)
true_gyros = np.array(true_gyros)

### Opti!

In [103]:
opti = cs.Opti()

quat = opti.variable(4, N)
vel = opti.variable(3, N)

# multiple shooting
for k in range(N-1):
    vector_SO3 = SO3Tangent(vel[:,k] * dt)
    rotation_SO3 = SO3(quat[:,k])
    opti.subject_to(quat[:,k + 1] == (vector_SO3 + rotation_SO3).as_quat())

Ca = 0
for k in range(N):
    R = SO3(quat[:,k]).as_matrix() 
    t = accs[k] - R @ acc0
    Ca += cs.sumsqr(t)

W = opti.variable(3,3)
opti.set_initial(W, np.eye(3))

Cg = cs.sumsqr(vel - W @ gyros.T)

opti.minimize(Cg + Ca)


In [104]:
# Set random initial guess
for k in range(N):
    opti.set_initial(quat[:,k], [0,0,0,1])

opti.set_initial(vel, gyros.T)

solver_options={"hessian_approximation":"limited-memory", "max_iter": 3000}
opti.solver("ipopt", {}, solver_options)
try:
    sol = opti.solve()
except:
    print("opti failed")

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.2.

Number of nonzeros in equality constraint Jacobian...:      288
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:       79
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:       36
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  2.4810076e+01 7.50e-01 1.83e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

In [105]:
q = opti.debug.value(quat).T
v = opti.debug.value(vel).T
AA = opti.debug.value(W)
AA

array([[-0.551,  0.412,  0.443],
       [-1.12 ,  0.246, -2.238],
       [-0.248, -0.238, -0.647]])

In [106]:
print(v)

[[-0.055 -2.866 -0.409]
 [-1.454  4.066  1.786]
 [-1.274  1.018  0.408]
 [ 0.048 -0.399 -0.516]
 [-0.063 -0.863 -0.706]
 [ 0.11   3.363  1.008]
 [ 0.764  0.44  -0.226]
 [ 0.112 -0.081 -0.167]
 [-0.468  0.056  0.118]
 [ 0.765 -0.022 -0.085]]


In [107]:
print(true_gyros)

[[-0.234 -0.234  1.579]
 [-1.913 -1.725 -0.562]
 [ 0.068 -1.425 -0.544]
 [ 1.852 -0.013 -1.058]
 [ 0.738  0.171 -0.116]
 [-1.763  0.324 -0.385]
 [ 0.331  0.976 -0.479]
 [ 1.004  0.362 -0.645]
 [ 0.087 -0.299  0.092]
 [-0.502  0.915  0.329]]


In [108]:
for k in range(N):
    R = SO3(q[k]).as_matrix().toarray()
    print(accs[k], R @ acc0)

[ 0.293  0.003 -0.956] [ 0.247  0.094 -0.963]
[ 0.061 -0.355  0.933] [-0.018 -0.165  0.886]
[-0.993 -0.08   0.082] [-0.947 -0.079 -0.066]
[-0.706  0.408  0.578] [-0.446  0.215  0.761]
[-0.558 -0.061  0.827] [-0.564  0.45   0.569]
[-0.153  0.976 -0.156] [-0.399  0.856  0.058]
[0.332 0.938 0.102] [0.474 0.795 0.166]
[0.861 0.154 0.485] [0.766 0.388 0.359]
[0.663 0.172 0.729] [0.779 0.207 0.455]
[ 0.785  0.579 -0.219] [0.759 0.49  0.245]
